# Supplementary Figures and Tables

Figures S1–S11 ordered by first reference in the revised manuscript.
S1, S3 are pre-made external figures (not generated in this notebook).

| Fig | Content | Source |
|-----|---------|--------|
| **S1** | Full-scale IQ distribution (SPARK & ASC) | External |
| **S2** | Temporal expression patterns across brain development | Inline |
| **S3** | Workflow for computing cell type mutation biases | External |
| **S4** | Specificity capping validation & noise from low expression | `Specificity_Cap_Analysis`, `ZINB_Simulation`, `Cap_Sensitivity` |
| **S5** | Impact of gene set size + real vs random expansion | `Number_Gene_Effect` |
| **S6** | Impact of genetic archeture / downsample mutations | `Number_Gene_Effect` |
| **S7** | Comprehensive mutation biases across brain cell types | Inline |
| **S8** | Negative controls (non-brain traits) & SCZ protective genes | `NegativeControl_BiasPlot`, `SCZ_Protective_BiasPlot` |
| **S9** | Impact of gene expression levels on ASD-SCZ bias correlation | `Similarity_ASD_SCZ.spec` |
| **S10** | ASD and SCZ mutation bias towards different cell type superclusters | Inline |
| **S11** | Mutation bias comparison: MGE & LAMP5-LHX6/Chandelier | Inline |
| **S12** | Comprehensive analysis of mutation biases in 22q11.2 deletion | Inline |

**Prerequisites:**
- Run `Number_Gene_Effect.ipynb` for gene sweep figures (S5)
- Run `Similarity_ASD_SCZ.spec.ipynb` for BrainSpan gene removal figure (S8)
- Run rebuttal notebooks for: specificity cap (S4), negative controls & SCZ protective (S7)

Tables S2–S11 follow the figures at the end.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import os
import io
import shutil
import subprocess
from contextlib import contextmanager

from pathlib import Path
import yaml
with open("/home/jw3514/Work/CellType_Psy/CellTypeBias_VIP/config/config.yaml") as f:
    _cfg = yaml.safe_load(f)
PROJ_DIR = Path(_cfg["ProjDIR"])
sys.path.insert(0, str(PROJ_DIR / "src"))
from CellType_PSY import *
from matplotlib.image import imread
from PIL import Image as PILImage

import matplotlib.font_manager as fm
font_path = '/usr/share/fonts/truetype/msttcorefonts/Arial.ttf'
fm.fontManager.addfont(font_path)
fm._load_fontmanager(try_read_cache=False)
plt.style.use('seaborn-v0_8-whitegrid')

FIG_DIR = str(PROJ_DIR / "results/figures/supp/") + "/"
os.makedirs(FIG_DIR, exist_ok=True)

REBUTTAL_FIG_DIR = str(PROJ_DIR / "results/figures/") + "/"

In [ ]:
# -- Utility functions --

@contextmanager
def save_panel(filepath, dpi=300):
    """Intercept plt.show() to save the figure before displaying inline."""
    _orig = plt.show
    plt.show = lambda *a, **kw: None
    try:
        yield
    finally:
        plt.show = _orig
        fig = plt.gcf()
        fig.savefig(filepath, dpi=dpi, bbox_inches='tight', transparent=True,
                    facecolor='none')
        plt.close(fig)
        from IPython.display import Image as IPImage, display as ipdisplay
        ipdisplay(IPImage(filename=filepath))


def assemble_panels(panel_paths, panel_labels, layout, figsize, out_path, dpi=300):
    """Assemble saved panel PNGs into a multi-panel composite figure."""
    nrows = max(r for r, _ in layout) + 1
    ncols = max(c.stop for _, c in layout)
    fig = plt.figure(figsize=figsize, dpi=dpi, facecolor='white')
    gs = fig.add_gridspec(nrows, ncols, hspace=0.08, wspace=0.08)

    for path, label, (row, col) in zip(panel_paths, panel_labels, layout):
        ax = fig.add_subplot(gs[row, col])
        img = imread(path)
        ax.imshow(img)
        ax.axis('off')
        ax.text(-0.02, 1.05, label, transform=ax.transAxes,
                fontsize=22, fontweight='bold', va='bottom', ha='right')

    fig.savefig(out_path, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"Saved composite: {out_path}")


def pdf_to_png(pdf_path, dpi=300):
    """Convert PDF to PNG using pdftoppm. Returns PNG path."""
    png_path = pdf_path.replace('.pdf', '.png')
    if os.path.exists(png_path):
        return png_path
    prefix = png_path.replace('.png', '')
    subprocess.run(['pdftoppm', '-png', '-r', str(dpi), '-singlefile',
                    pdf_path, prefix], check=True)
    print(f"Converted: {pdf_path} → {png_path}")
    return png_path


def copy_figure(src_path, fig_name, convert_pdf=True):
    """Copy a pre-generated figure to FIG_DIR with proper naming."""
    ext = os.path.splitext(src_path)[1]
    if ext == '.png':
        shutil.copy2(src_path, FIG_DIR + fig_name + ".png")
        img = PILImage.open(FIG_DIR + fig_name + ".png")
        img.save(FIG_DIR + fig_name + ".pdf", "PDF", resolution=300)
    elif ext == '.pdf':
        shutil.copy2(src_path, FIG_DIR + fig_name + ".pdf")
        if convert_pdf:
            pdf_to_png(FIG_DIR + fig_name + ".pdf")
    print(f"Saved: {FIG_DIR}{fig_name}.pdf")
    from IPython.display import Image as IPImage, display as ipdisplay
    png = FIG_DIR + fig_name + ".png"
    if os.path.exists(png):
        ipdisplay(IPImage(filename=png))

## Load data

In [ ]:
# Bias DataFrames
Bias_Save_Dir = str(PROJ_DIR / "results/main_results/random/Centering/") + "/"

ASD_All_Bias = pd.read_csv(Bias_Save_Dir + "ASD_All_bias_addP.csv", index_col=0)
SCZ_Bias = pd.read_csv(Bias_Save_Dir + "SCZ_bias_addP.csv", index_col=0)
HighIQ_ASD_Bias = pd.read_csv(Bias_Save_Dir + "ASD_HIQ_bias_addP.csv", index_col=0)
LowIQ_ASD_Bias = pd.read_csv(Bias_Save_Dir + "ASD_LIQ_bias_addP.csv", index_col=0)
X22q_Bias = pd.read_csv(Bias_Save_Dir + "22q_del_bias_addP.csv", index_col=0)
DDD_Bias = pd.read_csv(Bias_Save_Dir + "DDD_61_bias_addP.csv", index_col=0)
VNR_Pos_Bias = pd.read_csv(Bias_Save_Dir + "UKBB_VNR_Pos_bias_addP.csv", index_col=0)
VNR_Neg_Bias = pd.read_csv(Bias_Save_Dir + "UKBB_VNR_Neg_bias_addP.csv", index_col=0)
EDU_Pos_Bias = pd.read_csv(Bias_Save_Dir + "UKBB_EDU_Pos_bias_addP.csv", index_col=0)
EDU_Neg_Bias = pd.read_csv(Bias_Save_Dir + "UKBB_EDU_Neg_bias_addP.csv", index_col=0)

In [ ]:
# Pre-computed contrasts
CONTRAST_DIR = str(PROJ_DIR / "results/main_results/contrasts/") + "/"

ASD_SCZ_Contrast = pd.read_csv(CONTRAST_DIR + "ASD_woID_vs_SCZ_contrast.csv", index_col=0)
ASD_wID_SCZ_Contrast = pd.read_csv(CONTRAST_DIR + "ASD_wID_vs_SCZ_contrast.csv", index_col=0)
HIQ_LIQ_Contrast = pd.read_csv(CONTRAST_DIR + "ASD_woID_vs_ASD_wID_contrast.csv", index_col=0)
ASD_DDD_Contrast = pd.read_csv(CONTRAST_DIR + "ASD_woID_vs_DDD_contrast.csv", index_col=0)
SCZ_ASD_wID_Contrast = pd.read_csv(CONTRAST_DIR + "SCZ_vs_ASD_wID_contrast.csv", index_col=0)
VNR_Contrast = pd.read_csv(CONTRAST_DIR + "VNR_neg_vs_pos_contrast.csv", index_col=0)
EDU_Contrast = pd.read_csv(CONTRAST_DIR + "EDU_neg_vs_pos_contrast.csv", index_col=0)
all_contrasts_df = pd.read_csv(CONTRAST_DIR + "all_contrasts_fdr.csv")
brainspan_df = pd.read_csv(CONTRAST_DIR + "brainspan_expression.csv")

# Neuron-filtered contrasts
ASD_SCZ_Contrast_Neurons = ASD_SCZ_Contrast[ASD_SCZ_Contrast.index.isin(Neurons)]
HIQ_LIQ_Contrast_Neurons = HIQ_LIQ_Contrast[HIQ_LIQ_Contrast.index.isin(Neurons)]
ASD_DDD_Contrast_Neurons = ASD_DDD_Contrast[ASD_DDD_Contrast.index.isin(Neurons)]
VNR_Contrast_Neurons = VNR_Contrast[VNR_Contrast.index.isin(Neurons)]
EDU_Contrast_Neurons = EDU_Contrast[EDU_Contrast.index.isin(Neurons)]

EffLabel = "EFFECT"

---
## Figure S2 — Temporal Expression Patterns Across Brain Development

In [ ]:
palette = sns.color_palette("Set2", 5)

Time = ['mean_2A', 'mean_2B', 'mean_3A', 'mean_3B', 'mean_4', 'mean_5',
        'mean_6', 'mean_7', 'mean_8', 'mean_9', 'mean_10', 'mean_11']

labels_time = [
    "Embryonic", "Early fetal", "Early mid-fetal", "Late mid-fetal", "Late fetal",
    "Early infancy", "Late infancy", "Early Childhood", "Late childhood", "Adolescence",
    "Young adulthood", "Adulthood"
]

gene_set_styles = {
    "ASD w/o ID": dict(color=palette[0], marker='o', label="ASD genes"),
    "ASD with ID": dict(color=palette[1], marker='^', label="ASD with ID genes"),
    "SCZ": dict(color=palette[2], marker='s', label="SCZ genes"),
    "ALL": dict(color=palette[4], marker='*', label="ALL Genes", ls="--", markersize=12, alpha=0.8, zorder=2),
}

fig, ax = plt.subplots(dpi=300, figsize=(12, 8), constrained_layout=True)

xvals = np.arange(len(Time))
for gene_set, style in gene_set_styles.items():
    subset = brainspan_df[brainspan_df["gene_set"] == gene_set].sort_values("stage", key=lambda s: s.map({t: i for i, t in enumerate(Time)}))
    shift = {"ASD w/o ID": 0.05, "SCZ": -0.05}.get(gene_set, 0.0)
    ax.errorbar(
        x=xvals + shift,
        y=subset["mean"].values,
        yerr=subset["sem"].values,
        linewidth=2, markersize=style.get("markersize", 8),
        alpha=style.get("alpha", 0.95), zorder=style.get("zorder", 3),
        capsize=5, elinewidth=2, capthick=2,
        color=style["color"], marker=style["marker"], label=style["label"],
        ls=style.get("ls", "-"),
    )

ax.axvline(x=5, color='grey', ls="--", linewidth=1.5, zorder=1)

ax.set_xticks(xvals)
ax.set_xticklabels(labels_time, rotation=30, ha='right', fontsize=13, weight='bold')
ax.set_ylabel("Brain expression level (log10(RPKM))", fontsize=18, weight='bold')
ax.set_xlabel("Developmental stage", fontsize=16, weight='bold', labelpad=10)

sns.despine(ax=ax)
ax.grid(True, which='major', axis='y', linestyle='--', alpha=0.5, zorder=0)
ax.legend(loc="upper center", bbox_to_anchor=(0.7, 1), fancybox=True, shadow=True,
          ncol=2, fontsize=14, frameon=True, borderaxespad=0.5, title="Gene sets", title_fontsize=15)
ax.tick_params(axis='both', which='major', labelsize=13, width=1.5)

fig.savefig(FIG_DIR + "FigureS2.pdf", dpi=300, bbox_inches='tight', transparent=True, facecolor='none')
plt.show()
print(f"Saved: {FIG_DIR}FigureS2.pdf")

## Figure S4 — Specificity Capping Validation

**Moved (2026-07-01).** Fig S4 is now generated by the canonical
`notebooks_rebuttal/FigS4_Specificity_Cap.py` (2-panel: specificity
inflation vs library size + mutation-bias robustness to cap level).
The prior inline 8-panel version (incl. ZINB/TDEP exploratory panels)
was removed; those analyses now live in `dev_notebooks/specificity_cap/`.

## Figure S5 — Impact of Number of Genes on Mutation Bias Analysis

| Panel | Content | Source |
|-------|---------|--------|
| A | SCZ gene set size sweep | `Number_Gene_Effect.ipynb` |
| B | ASD w/o ID gene set size sweep | `Number_Gene_Effect.ipynb` |
| C | ASD with ID gene set size sweep | `Number_Gene_Effect.ipynb` |
| D | DD/ID gene set size sweep | `Number_Gene_Effect.ipynb` |
| E | Real vs random gene additions (SCZ, ASD w/ID, DDD) | `Number_Gene_Effect.ipynb` |

In [ ]:
# Load pre-computed gene sweep data
import pickle
import matplotlib.ticker as mticker

with open(PLOT_DATA_DIR + "gene_sweep_data.pkl", "rb") as f:
    gene_sweep = pickle.load(f)
with open(PLOT_DATA_DIR + "gene_expansion_data.pkl", "rb") as f:
    gene_expansion = pickle.load(f)


def plot_gene_set_correlation_inline(GeneIdx, Corr, GeneSig, ylabel_corr, ax1=None,
                                     Corr_unweighted=None, color_corr='#1f77b4',
                                     color_pval='#d62728', color_unweighted='#2ca02c'):
    """Plot gene set correlation and -log10(p-value) vs number of genes."""
    if ax1 is None:
        fig, ax1 = plt.subplots(figsize=(6, 4.5), dpi=120)
    else:
        fig = ax1.figure
    ax1.plot(GeneIdx, Corr, color=color_corr, linewidth=2, marker='o', markersize=4, label="Bias Correlation")
    # if Corr_unweighted is not None:
    #     ax1.plot(GeneIdx, Corr_unweighted, color=color_unweighted, linewidth=2, marker='^',
    #              markersize=4, label="Bias Correlation (Unweighted)", linestyle='--')
    ax1.set_xlabel("Number of Genes", fontsize=12)
    ax1.set_ylabel(ylabel_corr, color=color_corr, fontsize=12)
    ax1.tick_params(axis='y', labelcolor=color_corr)
    ax1.set_ylim(0, 1.01)
    ax1.yaxis.set_major_locator(mticker.MultipleLocator(0.2))
    ax1.grid(True, which='major', axis='y', linestyle='--', alpha=0.5, zorder=1)
    ax2 = ax1.twinx()
    ax2.plot(GeneIdx, GeneSig, color=color_pval, linewidth=2, marker='s', markersize=4,
             label=r"$-\log_{10}$(max p-value)", zorder=3)
    ax2.set_ylabel(r"$-\log_{10}$(max p-value)", color=color_pval, fontsize=12)
    ax2.tick_params(axis='y', labelcolor=color_pval)
    lines_1, labels_1 = ax1.get_legend_handles_labels()
    lines_2, labels_2 = ax2.get_legend_handles_labels()
    ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='lower left', fontsize=10, frameon=False)
    fig.tight_layout(pad=2)
    return fig, ax1, ax2

### Panel A–D — Gene Set Size Sweeps (2×2)

In [ ]:
panel_configs = [
    ('SCZ', gene_sweep['SCZ'], "Bias Correlation with Main SCZ Set", "A"),
    ('ASD_woID', gene_sweep['ASD_woID'], "Bias Correlation with Main ASD w/o ID Set", "B"),
    ('ASD_wID', gene_sweep['ASD_wID'], "Bias Correlation with Main ASD with ID Set", "C"),
    ('DDD', gene_sweep['DDD'], "Bias Correlation with Main DD/ID Set", "D"),
]

fig, axes_grid = plt.subplots(2, 2, figsize=(12, 8), dpi=120)
axes_flat = axes_grid.flatten()

for ax1, (key, d, ylabel, panel_label) in zip(axes_flat, panel_configs):
    color_corr, color_pval, color_uw = '#1f77b4', '#d62728', '#2ca02c'
    GeneIdx = gene_sweep['GeneIdx']

    ax1.plot(GeneIdx, d['Corr'], color=color_corr, linewidth=2, marker='o', markersize=3, label="Bias Correlation")
    SHOW_UNWEIGHTED = False
    if SHOW_UNWEIGHTED and d.get('Corr_unweighted'):
        ax1.plot(GeneIdx, d['Corr_unweighted'], color=color_uw, linewidth=2, marker='^',
                 markersize=3, label="Bias Correlation (Unweighted)", linestyle='--')
    ax1.set_xlabel("Number of Genes", fontsize=14)
    ax1.set_ylabel(ylabel, color=color_corr, fontsize=13)
    ax1.tick_params(axis='y', labelcolor=color_corr)
    ax1.set_ylim(0, 1.01)
    ax1.yaxis.set_major_locator(mticker.MultipleLocator(0.2))
    ax1.grid(True, which='major', axis='y', linestyle='--', alpha=0.5, zorder=1)
    ax1.tick_params(axis='both', labelsize=12)

    ax2 = ax1.twinx()
    ax2.plot(GeneIdx, d['GeneSig'], color=color_pval, linewidth=2, marker='s', markersize=3,
             label=r"$-\log_{10}$(max p-value)", zorder=3)
    ax2.set_ylabel(r"$-\log_{10}$(max p-value)", color=color_pval, fontsize=13)
    ax2.tick_params(axis='y', labelcolor=color_pval, labelsize=12)

    lines_1, labels_1 = ax1.get_legend_handles_labels()
    lines_2, labels_2 = ax2.get_legend_handles_labels()
    ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='lower left', fontsize=10, frameon=False)

    ax1.text(-0.02, 1.05, panel_label, transform=ax1.transAxes,
             fontsize=22, fontweight='bold', va='bottom', ha='right')

fig.tight_layout(pad=1.0, h_pad=1.0, w_pad=2.0)
fig.patch.set_alpha(0)
fig.savefig(FIG_DIR + "FigS5_ABCD.png", dpi=300, bbox_inches="tight", transparent=True, facecolor='none')
plt.show()

### Panel E–G — Real vs Random Gene Additions (SCZ, ASD with ID, DDD)

In [ ]:
total_genes = gene_expansion['total_genes']
disorders_exp = gene_expansion['disorders']
disorder_order = [("SCZ", "#ff7f0e", "E"), ("ASD", "#1f77b4", "F"), ("DDD", "#2ca02c", "G")]

fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=120, sharey=True)
for ax, (disorder_name, color, panel_label) in zip(axes, disorder_order):
    res = disorders_exp[disorder_name]
    ax.fill_between(total_genes, res["rand_lo"], res["rand_hi"],
                    color="#999999", alpha=0.25, label="Random genes (95% CI)")
    ax.plot(total_genes, res["rand_mean"], color="#999999", lw=2, ls="--",
            marker="s", markersize=4, label="Random genes (mean)")
    ax.plot(total_genes, res["real"], color=color, lw=2.5,
            marker="o", markersize=5, label=f"Ranked {disorder_name} genes", zorder=5)
    ax.axvline(61, color="gray", ls=":", lw=1.5, alpha=0.5)
    ax.set_xlabel("Total number of genes", fontsize=16)
    ax.set_title(disorder_name, fontweight="bold", fontsize=18)
    ax.text(-0.02, 1.05, panel_label, transform=ax.transAxes,
            fontsize=22, fontweight='bold', va='bottom', ha='right')
    ax.legend(fontsize=12, framealpha=0.8, loc="lower left")
    ax.set_ylim(0.3, 1.02)
    ax.tick_params(axis='both', labelsize=14)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
axes[0].set_ylabel("Spearmans' R with top-61 bias profile", fontsize=16)
fig.tight_layout()
fig.patch.set_alpha(0)
fig.savefig(FIG_DIR + "FigS5_EFG.png", dpi=300, bbox_inches='tight', transparent=True, facecolor='none')
plt.show()

---
## Figure S6 — Impact of Genetic Architecture / Downsample Mutations

**TODO:** Add panels from `Number_Gene_Effect.ipynb` (e.g., split-half bias comparison,
sliding window correlation decay, downsampling stability).

In [ ]:
# Placeholder — add S6 panels here
print("Figure S6: TODO — add genetic architecture / downsampling panels")

---
## Figure S7 — Comprehensive Mutation Biases Across Brain Cell Types in Psychiatric Disorders

| Panel | Content |
|-------|---------|
| A | ASD (all) |
| B | ASD w/o ID |
| C | ASD with ID |
| D | SCZ |
| E | DD/ID |

### Panel A — ASD

In [ ]:
with save_panel(FIG_DIR + "FigS7_A.png"):
    SuperClusterBias_BoxPlot(ASD_All_Bias, "ASD", NeuroOnly=False, sortby="mean", EffectCol="-logP", fdr_cut=0.05)

### Panel B — ASD w/o ID

In [ ]:
with save_panel(FIG_DIR + "FigS7_B.png"):
    SuperClusterBias_BoxPlot(HighIQ_ASD_Bias, "ASD w/o ID", NeuroOnly=False, sortby="mean", EffectCol="-logP", fdr_cut=0.1)

### Panel C — ASD with ID

In [ ]:
with save_panel(FIG_DIR + "FigS7_C.png"):
    SuperClusterBias_BoxPlot(LowIQ_ASD_Bias, "ASD with ID", NeuroOnly=False, sortby="mean", EffectCol="-logP")

### Panel D — SCZ

In [ ]:
with save_panel(FIG_DIR + "FigS7_D.png"):
    SuperClusterBias_BoxPlot(SCZ_Bias, "SCZ", NeuroOnly=False, sortby="mean", EffectCol="-logP")

### Panel E — DD/ID

In [ ]:
with save_panel(FIG_DIR + "FigS7_E.png"):
    SuperClusterBias_BoxPlot(DDD_Bias, "DD/ID", NeuroOnly=False, sortby="mean", EffectCol="-logP")

---
## Figure S8 — Negative Controls (Non-Brain Traits) & SCZ Protective Genes

| Panel | Content | Source |
|-------|---------|--------|
| A | Non-brain trait bias (HDL, Alanine, RBC, IBD) | `NegativeControl_BiasPlot.ipynb` |
| B | SCZ protective-direction (OR < 1) bias | `SCZ_Protective_BiasPlot.ipynb` |

CGE signal is specific to psychiatric risk genes; absent in non-brain traits
and inverted for protective-direction SCZ genes.

In [ ]:
# Load negative control and SCZ protective bias+pval from main pipeline results
_matched_dir = str(PROJ_DIR / "results/main_results/matched_WB_mean_phastCons_n_CDS_bases_Best1000/Centering/") + "/"

HDL_Pval = pd.read_csv(_matched_dir + "NegCtrl_HDL_bias_addP.csv", index_col=0)
IBD_Pval = pd.read_csv(_matched_dir + "NegCtrl_IBD_bias_addP.csv", index_col=0)
ALT_Pval = pd.read_csv(_matched_dir + "NegCtrl_Alanine_bias_addP.csv", index_col=0)
RBC_Pval = pd.read_csv(_matched_dir + "NegCtrl_RBC_bias_addP.csv", index_col=0)
SCZ_Protect_Pval = pd.read_csv(_matched_dir + "SCZ_protect_bias_addP.csv", index_col=0)

# Compute bias inline from gene weights (consistent with pipeline)
_gw_dir = str(PROJ_DIR / "dat/GeneWeights/") + "/"
_exp_mat = pd.read_csv(str(PROJ_DIR / _cfg['analysis_types']['Centering']), index_col=0)
_exp_mat.columns = _exp_mat.columns.astype(int)

HDL_Bias = AnnotateCTDat(HumanCT_AvgZ_Weighted(_exp_mat, Fil2Dict(_gw_dir + "NegCtrl_HDL.gw.csv")), Anno)
IBD_Bias = AnnotateCTDat(HumanCT_AvgZ_Weighted(_exp_mat, Fil2Dict(_gw_dir + "NegCtrl_IBD.gw.csv")), Anno)
ALT_Bias = AnnotateCTDat(HumanCT_AvgZ_Weighted(_exp_mat, Fil2Dict(_gw_dir + "NegCtrl_Alanine.gw.csv")), Anno)
RBC_Bias = AnnotateCTDat(HumanCT_AvgZ_Weighted(_exp_mat, Fil2Dict(_gw_dir + "NegCtrl_RBC.gw.csv")), Anno)
SCZ_Protect_Bias = AnnotateCTDat(HumanCT_AvgZ_Weighted(_exp_mat, Fil2Dict(_gw_dir + "SCZ.top61.protect.gw")), Anno)

### Panel A — Non-brain trait bias (EFFECT)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(32, 8), facecolor="none")
for ax, (bias_df, name) in zip(axes, [(HDL_Bias, "HDL Cholesterol"), (ALT_Bias, "Alanine AT"), (RBC_Bias, "Red Blood Cell"), (IBD_Bias, "IBD")]):
    ax.patch.set_alpha(0)
    SuperClusterBias_BoxPlot(bias_df, name, ax=ax)
fig.patch.set_alpha(0)
plt.tight_layout()
fig.savefig(FIG_DIR + "FigS8_A_bias.png", dpi=300, bbox_inches="tight", transparent=True)
plt.show()

### Panel B — Non-brain trait significance (-logP)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(32, 8), facecolor="none")
for ax, (pval_df, name) in zip(axes, [(HDL_Pval, "HDL Cholesterol"), (ALT_Pval, "Alanine AT"), (RBC_Pval, "Red Blood Cell"), (IBD_Pval, "IBD")]):
    ax.patch.set_alpha(0)
    SuperClusterBias_BoxPlot(pval_df, name, EffectCol="-logP", fdr_cut=0.1, ax=ax)
fig.patch.set_alpha(0)
plt.tight_layout()
fig.savefig(FIG_DIR + "FigS8_B_pval.png", dpi=300, bbox_inches="tight", transparent=True)
plt.show()

### Panel C — SCZ protective genes (EFFECT + significance)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8), facecolor="none")
axes[0].patch.set_alpha(0)
SuperClusterBias_BoxPlot(SCZ_Protect_Bias, "SCZ Protective (OR < 1)", ax=axes[0])
axes[1].patch.set_alpha(0)
SuperClusterBias_BoxPlot(SCZ_Protect_Pval, "SCZ Protective (OR < 1)", EffectCol="-logP", fdr_cut=0.1, ax=axes[1])
fig.patch.set_alpha(0)
plt.tight_layout()
fig.savefig(FIG_DIR + "FigS8_C_protect.png", dpi=300, bbox_inches="tight", transparent=True)
plt.show()

---
## Figure S9 — Impact of Gene Expression Levels on ASD-SCZ Bias Correlation

Pre-generated by Similarity_ASD_SCZ.spec notebook.

In [ ]:
from multiprocessing import Pool

with open(str(PROJ_DIR / "config/config.yaml")) as file:
    config = yaml.safe_load(file)

HGNC, ENSID2Entrez, GeneSymbol2Entrez, Entrez2Symbol = LoadGeneINFO()

# Load expression matrix
expression_matrix = config['analysis_types']['Centering']
HCT_Z2_MAT = pd.read_csv(str(PROJ_DIR / expression_matrix), index_col=0)
HCT_Z2_MAT.columns = HCT_Z2_MAT.columns.astype(int)

# Gene weights
ASD_GW = Fil2Dict(str(PROJ_DIR / "dat/GeneWeights/HIQ.top61.nopLI.LGD_Dmis_SameWeight.bgmr.gw"))
SCZ_GW = Fil2Dict(str(PROJ_DIR / "dat/GeneWeights/SCZ.top61.nopLI.LGD_Dmis_SameWeight.exclude_Mis2.gw"))

# BrainSpan expression
BrainSpan = pd.read_csv("/home/jw3514/Work/CellType_Psy/dat2/ExpMatch/BrainSpan.MatchDF.csv", index_col=0)

# gnomAD constraint (v4 primary, v2 fallback)
gnomad4 = pd.read_csv("/home/jw3514/Work/data/gnomad/gnomad.v4.0.constraint_metrics.tsv", sep="\t")
gnomad4 = gnomad4[(gnomad4["transcript"].str.contains('ENST')) & (gnomad4["mane_select"] == True)]
gnomad4["Entrez"] = gnomad4["gene"].map(GeneSymbol2Entrez).fillna(0).astype(int)
gnomad4 = gnomad4[gnomad4["Entrez"] != 0][["Entrez", "gene", "lof.pLI", "lof.z_score", "lof.oe_ci.upper"]].copy()

gnomad2 = pd.read_csv("/home/jw3514/Work/data/gnomad/gnomad.v2.1.1.lof_metrics.by_gene.txt", sep="\t")
gnomad2["Entrez"] = gnomad2["gene"].map(GeneSymbol2Entrez).fillna(0).astype(int)
gnomad2 = gnomad2[["Entrez", "gene", "pLI", "lof_z", "oe_lof_upper"]].copy()
gnomad2.columns = ["Entrez", "gene", "lof.pLI", "lof.z_score", "lof.oe_ci.upper"]
missing_in_v4 = set(gnomad2["Entrez"]) - set(gnomad4["Entrez"])
gnomad = pd.concat([gnomad4, gnomad2[gnomad2["Entrez"].isin(missing_in_v4)]], ignore_index=True)
gnomad = gnomad.drop_duplicates(subset="Entrez", keep="first").sort_values("lof.z_score", ascending=False)

# Annotate genes
def _annotate_gw(gw_dict, gnomad_df, brainspan_df):
    df = gnomad_df[gnomad_df["Entrez"].isin(gw_dict.keys())].copy()
    df["GW"] = df["Entrez"].map(gw_dict)
    df["BrainSpan"] = df["Entrez"].map(lambda x: brainspan_df.loc[x, "WB"] if x in brainspan_df.index else np.nan)
    return df

ASD_Genes = _annotate_gw(ASD_GW, gnomad, BrainSpan)
SCZ_Genes = _annotate_gw(SCZ_GW, gnomad, BrainSpan)

In [ ]:
# Pre-compute numpy structures for fast bias correlation
expr_np = HCT_Z2_MAT.values
expr_gene_set = set(HCT_Z2_MAT.index)
expr_gene_to_row = {g: i for i, g in enumerate(HCT_Z2_MAT.index)}
ct_cols = HCT_Z2_MAT.columns.values
neur_col_mask = np.array([int(c) in Neur_idx for c in ct_cols])

def fast_bias_corr(asd_entrez, asd_weights, scz_entrez, scz_weights):
    asd_rows = np.array([expr_gene_to_row[g] for g in asd_entrez])
    asd_bias = np.average(expr_np[asd_rows], axis=0, weights=asd_weights)
    scz_rows = np.array([expr_gene_to_row[g] for g in scz_entrez])
    scz_bias = np.average(expr_np[scz_rows], axis=0, weights=scz_weights)
    from scipy.stats import spearmanr
    r, _ = spearmanr(asd_bias[neur_col_mask], scz_bias[neur_col_mask])
    return r

def prepare_gene_arrays(genes_df):
    mask = genes_df["Entrez"].isin(expr_gene_set)
    return genes_df.loc[mask, "Entrez"].values, genes_df.loc[mask, "GW"].values

N_REMOVAL_STEPS = 31

def compute_removal_curve(asd_sorted_df, scz_sorted_df):
    asd_entrez, asd_weights = prepare_gene_arrays(asd_sorted_df)
    scz_entrez, scz_weights = prepare_gene_arrays(scz_sorted_df)
    return [fast_bias_corr(asd_entrez[i:], asd_weights[i:], scz_entrez[i:], scz_weights[i:]) for i in range(N_REMOVAL_STEPS)]

In [ ]:
# BrainSpan removal curves
ASD_by_BS = ASD_Genes.dropna(subset=["BrainSpan"]).sort_values("BrainSpan", ascending=False)
SCZ_by_BS = SCZ_Genes.dropna(subset=["BrainSpan"]).sort_values("BrainSpan", ascending=False)
Y_BS_high_first = compute_removal_curve(ASD_by_BS, SCZ_by_BS)

ASD_by_BS_rev = ASD_Genes.dropna(subset=["BrainSpan"]).sort_values("BrainSpan", ascending=True)
SCZ_by_BS_rev = SCZ_Genes.dropna(subset=["BrainSpan"]).sort_values("BrainSpan", ascending=True)
Y_BS_low_first = compute_removal_curve(ASD_by_BS_rev, SCZ_by_BS_rev)

# Load cached random null
CACHE_FILE = PROJ_DIR / "dat/Other/ASD_SCZ_RandomGeneRemoval_Null_v2.npy"
RandNull = np.load(CACHE_FILE)
rand_mean = RandNull.mean(axis=0)
rand_std = RandNull.std(axis=0)

X = list(range(N_REMOVAL_STEPS))

In [ ]:
fig, ax = plt.subplots(dpi=150, figsize=(9.5, 6), facecolor='none')
fig.patch.set_alpha(0.0)
ax.patch.set_alpha(0.0)

ax.plot(X, Y_BS_low_first, label="Remove lowest expressed genes first",
        color="red", linestyle='-', marker='o', markersize=8,
        markeredgecolor='black', markeredgewidth=1, zorder=10)
ax.plot(X, Y_BS_high_first, label="Remove highest expressed genes first",
        color="blue", linestyle='--', marker='s', markersize=8,
        markeredgecolor='black', markeredgewidth=1, zorder=10)
ax.errorbar(X, rand_mean, yerr=rand_std, fmt='-', color="grey", ecolor='grey',
            elinewidth=2, capsize=4, capthick=2, label="Random removal", zorder=5)

ax.set_xlabel("Number of Genes Removed", fontsize=25)
ax.set_ylabel("Mutation Bias Correlation", fontsize=25)
ax.legend(fontsize=18, loc='best', frameon=False)
ax.tick_params(axis='both', labelsize=15)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
fig.savefig(FIG_DIR + "FigureS9.png", dpi=300, transparent=True, bbox_inches='tight')
fig.savefig(FIG_DIR + "FigureS9.pdf", dpi=300, transparent=True, bbox_inches='tight')
plt.show()
print(f"Saved: {FIG_DIR}FigureS9.pdf")

---
## Figure S10 — ASD and SCZ Mutation Bias Towards Different Cell Type Superclusters

Individual cell type comparisons across disorders.

| Panel | Content |
|-------|---------|
| A | Hippo CA1-3 (ASD w/o ID vs SCZ) |
| B | Upper IT (ASD w/o ID vs SCZ) |
| C | Deep IT (ASD w/o ID vs SCZ) |
| D | Deep CT6b (ASD w/o ID vs SCZ) |
| E | ASD w/o ID vs ASD with ID bar plot |
| F | CGE: ASD with ID vs SCZ |
| F2 | MGE: ASD with ID vs SCZ |
| G | CGE: VNR+ vs VNR- |
| H | CGE: ASD w/o ID vs DD/ID |

### Panel A — Hippocampal CA1-3

In [ ]:
with save_panel(FIG_DIR + "FigS10_A.png"):
    CompareSingleCT(HighIQ_ASD_Bias, SCZ_Bias, "Hippocampal CA1-3", ASD_SCZ_Contrast,
                    "ASD w/o ID Mutation Bias", "SCZ Mutation Bias", efflabel=EffLabel, loc=(0.15, 0.12))

### Panel B — Upper-layer IT

In [ ]:
with save_panel(FIG_DIR + "FigS10_B.png"):
    CompareSingleCT(HighIQ_ASD_Bias, SCZ_Bias, "Upper-layer intratelencephalic", ASD_SCZ_Contrast,
                    "ASD w/o ID Mutation Bias", "SCZ Mutation Bias", efflabel=EffLabel, pval="Mann_Whitney_FDR", loc=(0.15, 0.12))

### Panel C — Deep-layer IT

In [ ]:
with save_panel(FIG_DIR + "FigS10_C.png"):
    CompareSingleCT(HighIQ_ASD_Bias, SCZ_Bias, "Deep-layer intratelencephalic", ASD_SCZ_Contrast,
                    "ASD w/o ID Mutation Bias", "SCZ Mutation Bias", loc=(0.1, 0.3), pval="Mann_Whitney_FDR")

### Panel D — Deep-layer CT6b

In [ ]:
with save_panel(FIG_DIR + "FigS10_D.png"):
    CompareSingleCT(HighIQ_ASD_Bias, SCZ_Bias, "Deep-layer corticothalamic and 6b", ASD_SCZ_Contrast,
                    "ASD w/o ID Mutation Bias", "SCZ Mutation Bias", efflabel=EffLabel, loc=(0.1, 0.05))

### Panel E — ASD w/o ID vs ASD with ID bar plot

In [ ]:
SCZ_ASD_wID_Contrast_Neurons = SCZ_ASD_wID_Contrast[SCZ_ASD_wID_Contrast.index.isin(Neurons)]

In [ ]:
with save_panel(FIG_DIR + "FigS10_E.png"):
    plot_bias_comparison(SCZ_ASD_wID_Contrast_Neurons, "SCZ", "ASD with ID",
                         p_test="Mann_Whitney_FDR", legend_anchor=(0.9, 1.0))

### Panel F — CGE: ASD with ID vs SCZ

In [ ]:
with save_panel(FIG_DIR + "FigS10_F.png"):
    CompareSingleCT(LowIQ_ASD_Bias, SCZ_Bias, "CGE interneuron", ASD_wID_SCZ_Contrast,
                    "ASD with ID Mutation Bias", "SCZ Mutation Bias", loc=(0.08, 0.21))

### Panel F2 — MGE: ASD with ID vs SCZ

In [ ]:
with save_panel(FIG_DIR + "FigS10_F2.png"):
    CompareSingleCT(LowIQ_ASD_Bias, SCZ_Bias, "MGE interneuron", ASD_wID_SCZ_Contrast,
                    "ASD with ID Mutation Bias", "SCZ Mutation Bias", loc=(0.08, 0.21))

### Panel G — CGE: VNR+ vs VNR-

In [ ]:
with save_panel(FIG_DIR + "FigS10_G.png"):
    CompareSingleCT(VNR_Pos_Bias, VNR_Neg_Bias, "CGE interneuron", VNR_Contrast,
                    "VNR + Mutation Bias", "VNR - Mutation Bias", loc=(0.0, -0.05))

### Panel H — CGE: ASD w/o ID vs DD/ID

In [ ]:
with save_panel(FIG_DIR + "FigS10_H.png"):
    CompareSingleCT(HighIQ_ASD_Bias, DDD_Bias, "CGE interneuron", ASD_DDD_Contrast,
                    "ASD w/o ID Mutation Bias", "DD/ID Mutation Bias", loc=(0.1, 0.05))

---
## Figure S11 — Mutation Bias Comparison Across Psychiatric Disorders for MGE & LAMP5-LHX6/Chandelier

| Panel | Content |
|-------|---------|
| A | MGE interneuron |
| B | LAMP5-LHX6 and Chandelier |

In [ ]:
datasets = {
    'ASD w/o ID': HighIQ_ASD_Bias,
    'ASD with ID': LowIQ_ASD_Bias,
    'VNR+': VNR_Pos_Bias,
    'VNR-': VNR_Neg_Bias,
    'DD/ID': DDD_Bias,
    'SCZ': SCZ_Bias
}
TestPairs = [("VNR+", "VNR-"), ("ASD w/o ID", "SCZ"), ("ASD w/o ID", "ASD with ID"),
             ("SCZ", "ASD with ID"), ("ASD w/o ID", "DD/ID")]

### Panel A — MGE interneuron

In [ ]:
with save_panel(FIG_DIR + "FigS11_A.png"):
    plot_mutation_bias_comparison_V2("MGE interneuron", datasets, Anno, all_contrasts_df, TestPairs=TestPairs)

### Panel B — LAMP5-LHX6 and Chandelier

In [ ]:
with save_panel(FIG_DIR + "FigS11_B.png"):
    plot_mutation_bias_comparison_V2("LAMP5-LHX6 and Chandelier", datasets, Anno, all_contrasts_df, TestPairs=TestPairs)

---
## Figure S12 — Comprehensive Analysis of Mutation Biases Across Brain Cell Types in 22q11.2 Deletion

In [ ]:
with save_panel(FIG_DIR + "FigureS12.png"):
    SuperClusterBias_BoxPlot(X22q_Bias, "22q11.2", NeuroOnly=False, sortby="mean", EffectCol="-logP", fdr_cut=0.1)

img = PILImage.open(FIG_DIR + "FigureS12.png")
img.save(FIG_DIR + "FigureS12.pdf", "PDF", resolution=300)
print(f"Saved: {FIG_DIR}FigureS12.pdf")

---
# Supplementary Tables

---
## Supplementary Tables S2-S7: Cluster-level biases

In [ ]:
import openpyxl

def prepare_bias_table_v1(df):
    """Prepare a bias DataFrame for supplementary table output."""
    df_out = df.copy(deep=True)
    df_out.index.name = "Cluster"
    columns_to_keep = [
        "EFFECT", "P-value", "q-value", "Class", "Supercluster", "Subtype", "Neurotransmitter",
        "Top three regions", "Top three dissections", "Number of cells"]
    df_out = df_out[columns_to_keep]
    df_out.rename(columns={"EFFECT": "Bias"}, inplace=True)
    df_out = df_out.sort_values(by=["P-value", "Bias"], ascending=[True, False])
    return df_out

def prepare_bias_table_v2(BiasPos, BiasNeg, Name1, Name2):
    df_out = pd.DataFrame()
    df_out["Bias {}".format(Name1)] = BiasPos["EFFECT"]
    df_out["P-value {}".format(Name1)] = BiasPos["P-value"]
    df_out["q-value {}".format(Name1)] = BiasPos["q-value"]
    df_out["Bias {}".format(Name2)] = BiasNeg["EFFECT"]
    df_out["P-value {}".format(Name2)] = BiasNeg["P-value"]
    df_out["q-value {}".format(Name2)] = BiasNeg["q-value"]
    df_out["Class"] = BiasPos["Class"]
    df_out["Supercluster"] = BiasPos["Supercluster"]
    df_out["Subtype"] = BiasPos["Subtype"]
    df_out["Neurotransmitter"] = BiasPos["Neurotransmitter"]
    df_out["Top three regions"] = BiasPos["Top three regions"]
    df_out["Top three dissections"] = BiasPos["Top three dissections"]
    df_out["Number of cells"] = BiasPos["Number of cells"]
    df_out.index.name = "Cluster"
    df_out = df_out.sort_values(by=["P-value {}".format(Name2), "Bias {}".format(Name2)], ascending=[True, False])
    return df_out

SCZ_Bias_toST = prepare_bias_table_v1(SCZ_Bias)
HighIQ_ASD_Bias_toST = prepare_bias_table_v1(HighIQ_ASD_Bias)
LowIQ_ASD_Bias_toST = prepare_bias_table_v1(LowIQ_ASD_Bias)
X22q_Bias_toST = prepare_bias_table_v1(X22q_Bias)
DDD_Bias_toST = prepare_bias_table_v1(DDD_Bias)
UKBB_VNR_Bias_toST = prepare_bias_table_v2(VNR_Pos_Bias, VNR_Neg_Bias, "VNR+", "VNR-")

In [ ]:
SuppTabOutDir = str(PROJ_DIR / "dat/suppl.data/") + "/"
excel_path = SuppTabOutDir + "SupTab_VIP.xlsx"

# Remove existing data sheets (keep Table_of_contents and experiment sheets)
wb = openpyxl.load_workbook(excel_path)
sheets_to_remove = [s for s in wb.sheetnames if s not in [
    "Table_of_contents",
    "Mouse Experiment Statistical detail  Statistical detail summary.",
    "Experiement Statistical detail "
]]
for s in sheets_to_remove:
    wb.remove(wb[s])
wb.save(excel_path)
wb.close()

In [ ]:
with pd.ExcelWriter(excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    Name_Dict = {
        "SCZ_Bias": "Table_S2_Cluster_Bias_SCZ",
        "HighIQ_ASD_Bias": "Table_S3_Cluster_Bias_ASD_woID",
        "LowIQ_ASD_Bias": "Table_S4_Cluster_Bias_ASD_ID",
        "X22q_Bias": "Table_S5_Cluster_Bias_22q11.2",
        "DDD_Bias": "Table_S6_Cluster_Bias_DD",
        "VNR_Bias": "Table_S7_Cluster_Bias_VNR",
    }
    DF_list = [
        ("SCZ_Bias", SCZ_Bias_toST),
        ("HighIQ_ASD_Bias", HighIQ_ASD_Bias_toST),
        ("LowIQ_ASD_Bias", LowIQ_ASD_Bias_toST),
        ("X22q_Bias", X22q_Bias_toST),
        ("DDD_Bias", DDD_Bias_toST),
        ("VNR_Bias", UKBB_VNR_Bias_toST),
    ]
    for df_name, DF in DF_list:
        sheet_name = Name_Dict.get(df_name, df_name)
        DF.to_excel(writer, sheet_name=sheet_name)

print("Tables S2-S7 written.")

---
## Supplementary Tables S8-S11: SuperCluster bias contrasts

In [ ]:
def process_BiasContrast_df(df):
    columns_to_drop = ["Wilcoxon_P", "Wilcoxon_FDR", "Bonferroni_P"]
    df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])
    disorder_name_1 = df.columns[0].replace("Bias_", "")
    disorder_name_2 = df.columns[1].replace("Bias_", "")
    df.rename(columns={"Bias_Diff": "Bias_Diff_{}_{}".format(disorder_name_1, disorder_name_2)}, inplace=True)
    return df

ASD_SCZ_Contrast_toST = process_BiasContrast_df(ASD_SCZ_Contrast_Neurons)
HIQ_LIQ_Contrast_toST = process_BiasContrast_df(HIQ_LIQ_Contrast_Neurons)
DDD_ASD_Contrast_toST = process_BiasContrast_df(ASD_DDD_Contrast_Neurons)
VNR_Contrast_toST = process_BiasContrast_df(VNR_Contrast_Neurons)

In [ ]:
with pd.ExcelWriter(excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    Name_Dict = {
        "BiasContrast_ASD_SCZ": "Table_S8_BiasContrast_ASD_SCZ",
        "BiasContrast_HIQ_LIQ": "Table_S9_BiasContrast_HIQ_LIQ",
        "BiasContrast_HIQ_DDD": "Table_S10_BiasContrast_HIQ_DDD",
        "BiasContrast_VNR": "Table_S11_BiasContrast_VNR",
    }
    DF_list = [
        ("BiasContrast_ASD_SCZ", ASD_SCZ_Contrast_toST),
        ("BiasContrast_HIQ_LIQ", HIQ_LIQ_Contrast_toST),
        ("BiasContrast_HIQ_DDD", DDD_ASD_Contrast_toST),
        ("BiasContrast_VNR", VNR_Contrast_toST),
    ]
    for df_name, DF in DF_list:
        sheet_name = Name_Dict.get(df_name, df_name)
        DF.to_excel(writer, sheet_name=sheet_name)

print("Tables S8-S11 written.")